In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt


# datafile_path = '/home/ricardo/ownCloud/ambev_libs/SVM_ambev/svm_full/full_data_ceramic/'
datafile_path = '/home/ricardo/Downloads/01_09_2026/'
# 1) Carregar os dados
# datafile_path = '/home/ricardo/Dropbox/pyhton_ML/erva/'

spectra = pd.read_csv(
    datafile_path + 'svm_coffee_full_exp_snv.dat',
    # datafile_path + 'svm_coffee_full_exp_group_snv.dat',  # 
    delimiter=';',
    header=0,
    index_col=None
)

# 2) Montar X e y
# Cada coluna (exceto 'wavelength') vira uma amostra
features = spectra.drop('wavelength', axis=1).T
samples = features.index

# Definição dos nomes das classes reais
class_names = unique_names

# Função para atribuir rótulos com base no nome da amostra
def assign_label(sample):
    sample_lower = sample.lower()
    for idx, name in enumerate(class_names):
        if name.lower() in sample_lower:
            return idx
    return -1  # Caso não encontre correspondência

y = np.array([assign_label(s) for s in samples])

# Verifica se houve alguma amostra sem rótulo identificado
if np.any(y == -1):
    print("Atenção: Algumas amostras não foram categorizadas corretamente:")
    print(samples[y == -1])

# 3) NÃO escalar aqui para evitar leakage; apenas pegue os valores
X = features.values  # sem StandardScaler.fit_transform em todo o conjunto [page:0]

# 4) Split treino/teste (antes de qualquer pré-processamento) [page:0]
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    stratify=y,
    random_state=42
)

# 5) Treinar SVM com diferentes kernels
kernels = ['linear', 'poly', 'rbf', 'sigmoid']
# kernels = ['linear']

for kernel in kernels:
    # Pipeline: primeiro scaler, depois SVM, para que o scaler seja ajustado
    # somente nos dados de treino (no split e em cada fold da CV) [page:0]
    svm_clf = Pipeline([
        ('scaler', StandardScaler()),
        ('svc', SVC(kernel=kernel, C=0.1, random_state=42))
    ])

    # Treino: fit usa apenas X_train / y_train [page:0]
    svm_clf.fit(X_train, y_train)

    # 6) Previsão e avaliação no teste (transform aplicado internamente) [page:0]
    y_pred = svm_clf.predict(X_test)

    disp = ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred,
        display_labels=class_names,
        # cmap=plt.cm.Blues
        cmap=plt.cm.Reds
    )
    plt.title(f'Matriz de Confusão (SVM - Kernel: {kernel})', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=16)
    plt.ylabel('True Label', fontsize=16)

    plt.xticks(fontsize=16, rotation=0)
    plt.yticks(fontsize=16)

    for text in disp.text_.flatten():
        text.set_fontsize(16)

    # plt.tight_layout()
    # plt.show()

    # Gera um nome de arquivo único para cada kernel
    file_name = f"plot_full_StratifiedKFold_{kernel}.png"
    file_path = datafile_path + file_name
    
    plt.savefig(file_path, dpi=1200, bbox_inches='tight')
    

    
    # plt.savefig(datafile_path + "plot_full_bkg_StratifiedKFold.png", dpi=1200, bbox_inches='tight')
    #
    #
    print(f"\nClassification Report (SVM - Kernel: {kernel}):")
    print(classification_report(y_test, y_pred, target_names=unique_names))

    # 7) Validação Cruzada
    # A CV também usa o pipeline, então em cada fold o scaler é ajustado
    # somente no treino daquele fold, evitando leakage na etapa de pré-processamento. [page:0]
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    # cv_scores = cross_val_score(svm_clf, X, y, cv=skf)
    cv_scores = cross_val_score(svm_clf, X_train, y_train, cv=skf)
    print(f"\n Accuracy CV (10 - fold) - Kernel {kernel}:", cv_scores)
    print(f" Mean CV - Kernel {kernel}:", cv_scores.mean())

# 6.1) print the number of vectors of the SVM
# Como o SVC está dentro do pipeline, recuperamos o passo 'svc' [page:0]
svc_step = svm_clf.named_steps['svc']
print(svc_step.n_support_)